# 03 - Quantum Fourier Transform (QFT)

**Module 3 - Quantum Algorithm Demonstration Platform**

The QFT is the quantum analogue of the discrete Fourier transform and the engine behind phase estimation and Shor's algorithm. It maps between the computational basis and the **phase/frequency basis**, encoding information into the relative phases of qubits.

To prove the implementation is correct we apply **QFT followed by inverse-QFT**, which is mathematically the identity: we must recover the original input state with very high probability. We call the platform's `demonstrate_qft()` from `backend/app/algorithms/qft.py`.

**Business relevance:** the QFT underpins quantum phase estimation, which is the subroutine that threatens RSA/ECC via Shor's algorithm - directly motivating the post-quantum cryptography module (Module 5).

In [1]:
# Make the platform's backend package importable from this notebook.
# The notebooks call the SAME code the FastAPI app runs, so results here
# and results in the deployed platform are guaranteed identical.
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..', 'backend')))
print('Backend path wired. Python', sys.version.split()[0])

Backend path wired. Python 3.14.4


In [2]:
from app.algorithms.qft import demonstrate_qft, build_qft_circuit

num_qubits = 3
input_state = '110'
print('Forward QFT circuit on', num_qubits, 'qubits:')
print(build_qft_circuit(num_qubits).draw(output='text'))

Forward QFT circuit on 3 qubits:
                                          ┌───┐   
q_0: ──────■──────────────────────■───────┤ H ├─X─
           │                ┌───┐ │P(π/2) └───┘ │ 
q_1: ──────┼────────■───────┤ H ├─■─────────────┼─
     ┌───┐ │P(π/4)  │P(π/2) └───┘               │ 
q_2: ┤ H ├─■────────■───────────────────────────X─
     └───┘                                        


In [3]:
result = demonstrate_qft(num_qubits=num_qubits, input_state=input_state, shots=1024)

print('Input state                     :', result['input_state'])
print('Recovered-correctly probability :', result['recovered_correctly_probability'])
print()
print(result['explanation'])

Input state                     : 110
Recovered-correctly probability : 1.0

QFT followed by inverse-QFT is mathematically the identity operation. A high probability of recovering the original input_state confirms the QFT implementation is correct.


### Full QFT -> inverse-QFT circuit

A recovered-correctly probability at or near 1.0 confirms the round trip is the identity, i.e. the QFT implementation is correct.

In [4]:
print(result['circuit_diagram_text'])

                                               ┌───┐      ┌───┐               »
q_0: ───────────■──────────────────────■───────┤ H ├─X──X─┤ H ├─■─────────────»
     ┌───┐      │                ┌───┐ │P(π/2) └───┘ │  │ └───┘ │P(-π/2) ┌───┐»
q_1: ┤ X ├──────┼────────■───────┤ H ├─■─────────────┼──┼───────■────────┤ H ├»
     ├───┤┌───┐ │P(π/4)  │P(π/2) └───┘               │  │                └───┘»
q_2: ┤ X ├┤ H ├─■────────■───────────────────────────X──X─────────────────────»
     └───┘└───┘                                                               »
c: 3/═════════════════════════════════════════════════════════════════════════»
                                                                              »
«                                 ┌─┐   
«q_0: ───────────■────────────────┤M├───
«                │        ┌─┐     └╥┘   
«q_1: ─■─────────┼────────┤M├──────╫────
«      │P(-π/2)  │P(-π/4) └╥┘┌───┐ ║ ┌─┐
«q_2: ─■─────────■─────────╫─┤ H ├─╫─┤M├
«                          ║ └───┘

In [5]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

probs = result['probabilities']
states = sorted(probs)
colors = ['#00b894' if s == input_state else '#b2bec3' for s in states]
plt.figure(figsize=(6,3))
plt.bar(states, [probs[s] for s in states], color=colors)
plt.title(f"QFT then inverse-QFT recovers the input '{input_state}' (green)")
plt.xlabel('Bitstring'); plt.ylabel('Probability')
plt.tight_layout(); plt.show()

/tmp/ipykernel_29990/3729765131.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()
